# Modeling ประมาณองค์ประกอบร่างกายจากค่าวัด

ใช้ชุดข้อมูลจาก `src/real_label.py` (NHANES 2017-2018 ผู้ใหญ่ 18-59 ปีที่มีผล DXA ครบ) ประมาณ Fat % กับมวลกล้ามเนื้อ (Lean mass, ALM, Trunk lean) จากค่าที่วัดง่าย โดยไม่ต้องสแกน DXA แล้วต่อยอดไปคัดกรองคนที่มีมวลกล้ามเนื้อต่ำ

In [1]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold, cross_val_predict, cross_val_score, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier
warnings.filterwarnings("ignore")

os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
sys.path.insert(0, "src")
import real_label as rl
df, n_adults = rl.build()
df.to_csv(rl.OUT, index=False)
BASIC, CIRC, LIFE = rl.BASIC, rl.CIRC, rl.LIFE
FEAT = BASIC + CIRC
TARGETS = ["FAT_PCT", "LEAN_KG", "ALM_KG", "TRUNK_KG"]
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df.LOW_HT2)
print(f"adults 18-59: {n_adults} -> with DXA + body measures: {len(df)} | train {len(train)} / test {len(test)}")

def score(y, p):
    return {"R2": r2_score(y, p), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p)))}

def ridge():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=[0.1, 1, 10, 100]))

summary = {"n_adults": n_adults, "n": len(df), "n_train": len(train), "n_test": len(test),
           "prevalence": {"LOW_HT2": float(df.LOW_HT2.mean()), "LOW_FNIH": float(df.LOW_FNIH.mean()), "both": int((df.LOW_FNIH & df.LOW_HT2).sum())}}

adults 18-59: 3515 -> with DXA + body measures: 2310 | train 1848 / test 462


## 1. DXA หายในคนที่น้ำหนักมาก (selection bias)

In [2]:
bins = pd.cut(df.BMXWT, [0, 60, 80, 100, 120, 300], labels=["<60", "60-80", "80-100", "100-120", ">=120"])
print(df.groupby(bins, observed=True).agg(n=("SEQN", "size"), fat_pct=("FAT_PCT", "mean"), lean=("LEAN_KG", "mean")).round(1))
print("low muscle (ALM/ht^2):", int(df.LOW_HT2.sum()), "| (ALM/BMI):", int(df.LOW_FNIH.sum()), "| both:", int((df.LOW_FNIH & df.LOW_HT2).sum()))

           n  fat_pct  lean
BMXWT                      
<60      413     30.0  35.7
60-80    919     31.8  46.1
80-100   639     33.9  56.6
100-120  240     36.9  66.2
>=120     99     40.5  75.2
low muscle (ALM/ht^2): 233 | (ALM/BMI): 234 | both: 23


## 2. ก่อนเทรน ทายค่าเฉลี่ยกับใช้สูตร Deurenberg (1991)

สูตรคือ `Fat% = 1.20×BMI + 0.23×อายุ − 10.8×(ชาย=1) − 5.4` ใช้เป็นเส้นฐานไว้เทียบกับ model

In [3]:
before = {t: score(test[t], np.full(len(test), train[t].mean())) for t in TARGETS}
deur = 1.20 * test.BMXBMI + 0.23 * test.RIDAGEYR - 10.8 * (test.RIAGENDR == 1) - 5.4
deur_score = score(test.FAT_PCT, deur)
display(pd.DataFrame(before).T.round(3)); print("Deurenberg (Fat %):", {k: round(v, 3) for k, v in deur_score.items()})
summary["before"], summary["deurenberg"] = before, deur_score

,R2,MAE,RMSE
FAT_PCT,-0.004,7.199,8.687
LEAN_KG,-0.000,10.485,12.789
ALM_KG,-0.000,5.260,6.465
TRUNK_KG,-0.000,5.104,6.258


Deurenberg (Fat %): {'R2': 0.529, 'MAE': 4.612, 'RMSE': 5.949}


## 3. ลอง 3 โมเดล Ridge, Random Forest, XGBoost

ปรับค่าตั้งของแต่ละโมเดลด้วย 5-fold CV บนชุดสอนอย่างเดียว แล้ววัดบน Test พร้อม CV ± std และเวลาที่ใช้

In [4]:
xgb_grid = {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 4], "learning_rate": [0.03, 0.05, 0.1]}
rf_grid = {"n_estimators": [200, 400], "max_depth": [6, 10, None], "min_samples_leaf": [1, 3, 5]}
kf = KFold(5, shuffle=True, random_state=42)
tuned_xgb, mc_rows, models3, best = {}, [], {}, {}
for t in TARGETS:
    gx = GridSearchCV(XGBRegressor(random_state=42), xgb_grid, cv=5, scoring="r2", n_jobs=-1).fit(train[FEAT], train[t])
    gr = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), rf_grid, cv=5, scoring="r2", n_jobs=1).fit(train[FEAT], train[t])
    tuned_xgb[t] = gx.best_estimator_
    best[t] = {"XGBoost": gx.best_params_, "Random Forest": gr.best_params_}
    models3[t] = {"Ridge": ridge(), "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1, **gr.best_params_),
                  "XGBoost": XGBRegressor(random_state=42, **gx.best_params_)}
    for name, m in models3[t].items():
        cv = cross_val_score(m, train[FEAT], train[t], cv=kf, scoring="r2")
        t0 = time.time(); m.fit(train[FEAT], train[t]); fit_s = time.time() - t0
        t0 = time.time(); p = m.predict(test[FEAT]); pred_ms = (time.time() - t0) * 1e6 / len(test)
        if name == "Ridge": best[t]["Ridge alpha"] = float(m[-1].alpha_)
        mc_rows.append({"Target": t, "Model": name, **{"Test " + k: v for k, v in score(test[t], p).items()},
                        "CV R2": cv.mean(), "CV std": cv.std(), "Fit s": fit_s, "Predict ms/1000": pred_ms})
df_mc = pd.DataFrame(mc_rows); summary["best_params"] = best
print("ค่าที่ชนะจาก GridSearchCV (Ridge คือ alpha จาก RidgeCV)")
for t in TARGETS: print(t, best[t])
df_mc.round(3)

ค่าที่ชนะจาก GridSearchCV (Ridge คือ alpha จาก RidgeCV)
FAT_PCT {'XGBoost': {'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}, 'Ridge alpha': 0.1}
LEAN_KG {'XGBoost': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}, 'Ridge alpha': 0.1}
ALM_KG {'XGBoost': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400}, 'Ridge alpha': 0.1}
TRUNK_KG {'XGBoost': {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}, 'Random Forest': {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 400}, 'Ridge alpha': 0.1}


,Target,Model,Test R2,Test MAE,Test RMSE,CV R2,CV std,Fit s,Predict ms/1000
0,FAT_PCT,Ridge,0.824,2.879,3.634,0.825,0.017,0.006,4.329
1,FAT_PCT,Random Forest,0.835,2.753,3.527,0.845,0.007,0.612,185.118
2,FAT_PCT,XGBoost,0.837,2.750,3.498,0.846,0.010,0.291,11.929
3,LEAN_KG,Ridge,0.949,2.240,2.898,0.953,0.004,0.006,0.000
4,LEAN_KG,Random Forest,0.948,2.246,2.908,0.948,0.006,0.669,198.448
5,LEAN_KG,XGBoost,0.951,2.187,2.845,0.950,0.005,0.412,13.053
6,ALM_KG,Ridge,0.928,1.340,1.739,0.931,0.010,0.006,2.168
7,ALM_KG,Random Forest,0.925,1.343,1.773,0.918,0.012,0.970,214.161
8,ALM_KG,XGBoost,0.930,1.301,1.714,0.924,0.011,0.424,8.672
9,TRUNK_KG,Ridge,0.928,1.288,1.678,0.933,0.004,0.007,2.160


## 4. ทดสอบกับคนน้ำหนักมาก

สอนด้วยคนต่ำกว่า 90 kg แล้วทายคน 100 kg ขึ้นไป จำลองสถานการณ์จริงที่ DXA มักหายในคนน้ำหนักมาก

In [5]:
lo, hi = df[df.BMXWT < 90], df[df.BMXWT >= 100]
ex_rows = []
for t in TARGETS:
    for name, m in models3[t].items():
        p = clone(m).fit(lo[FEAT], lo[t]).predict(hi[FEAT])
        ex_rows.append({"Target": t, "Model": name, "n heavy": len(hi), "Heavy MAE": mean_absolute_error(hi[t], p), "Heavy bias": float(np.mean(p - hi[t]))})
df_ex = pd.DataFrame(ex_rows); df_ex.round(2)

,Target,Model,n heavy,Heavy MAE,Heavy bias
0,FAT_PCT,Ridge,339,4.70,4.32
1,FAT_PCT,Random Forest,339,3.71,-2.59
2,FAT_PCT,XGBoost,339,3.47,-1.97
3,LEAN_KG,Ridge,339,3.16,1.12
4,LEAN_KG,Random Forest,339,12.98,-12.97
5,LEAN_KG,XGBoost,339,14.08,-14.06
6,ALM_KG,Ridge,339,1.95,0.91
7,ALM_KG,Random Forest,339,5.19,-5.14
8,ALM_KG,XGBoost,339,6.22,-6.17
9,TRUNK_KG,Ridge,339,1.77,-0.02


## 5. เลือกโมเดล

| โจทย์ | เลือก | เหตุผล |
|---|---|---|
| Fat % | **XGBoost** | แม่นสุดทั้ง CV และ Test และพลาดน้อยสุดในคนหนัก |
| Lean mass, ALM, Trunk lean | **Ridge** | แม่นเท่าโมเดลอื่น (ต่างไม่เกิน 0.005 ใน CV) แต่ง่ายสุด เร็วสุด ตีความได้ และทนคนหนักได้ดีกว่ามากกว่า 4 เท่า |
| ไม่เลือก | Random Forest | ไม่ชนะข้อไหนเลย ช้ากว่า ไม่แม่นกว่า และ extrapolate ไม่ได้ |

In [6]:
final_name = {"FAT_PCT": "XGBoost", "LEAN_KG": "Ridge", "ALM_KG": "Ridge", "TRUNK_KG": "Ridge"}
def make_final(t):
    return clone(models3[t][final_name[t]])
final = {t: make_final(t).fit(train[FEAT], train[t]) for t in TARGETS}
summary["model_comparison"] = {"regression": df_mc.to_dict("records"), "extrapolation": df_ex.to_dict("records"), "final": final_name}
tbl = df_mc.merge(df_ex, on=["Target", "Model"])
tbl["Chosen"] = [final_name[r.Target] == r.Model for r in tbl.itertuples()]
tbl[["Target", "Model", "Test R2", "Test MAE", "CV R2", "Heavy MAE", "Predict ms/1000", "Chosen"]].round(3)

,Target,Model,Test R2,Test MAE,CV R2,Heavy MAE,Predict ms/1000,Chosen
0,FAT_PCT,Ridge,0.824,2.879,0.825,4.704,4.329,False
1,FAT_PCT,Random Forest,0.835,2.753,0.845,3.708,185.118,False
2,FAT_PCT,XGBoost,0.837,2.750,0.846,3.472,11.929,True
3,LEAN_KG,Ridge,0.949,2.240,0.953,3.156,0.000,True
4,LEAN_KG,Random Forest,0.948,2.246,0.948,12.984,198.448,False
5,LEAN_KG,XGBoost,0.951,2.187,0.950,14.076,13.053,False
6,ALM_KG,Ridge,0.928,1.340,0.931,1.945,2.168,True
7,ALM_KG,Random Forest,0.925,1.343,0.918,5.194,214.161,False
8,ALM_KG,XGBoost,0.930,1.301,0.924,6.224,8.672,False
9,TRUNK_KG,Ridge,0.928,1.288,0.933,1.773,2.160,True


## 6. ผลสุดท้าย ก่อนเทรนเทียบหลังเทรน (โมเดลที่เลือก) พร้อมช่วงความเชื่อมั่น (bootstrap 1,000 รอบ)

In [7]:
after = {t: score(test[t], final[t].predict(test[FEAT])) for t in TARGETS}
rng = np.random.default_rng(42); boot = {}
preds = {t: final[t].predict(test[FEAT]) for t in TARGETS}
for t in TARGETS:
    r2s, maes, gain = [], [], []
    for _ in range(1000):
        i = rng.integers(0, len(test), len(test)); y, p = test[t].values[i], preds[t][i]
        r2s.append(r2_score(y, p)); maes.append(mean_absolute_error(y, p))
        gain.append(1 - mean_absolute_error(y, p) / mean_absolute_error(y, np.full(len(y), train[t].mean())))
    boot[t] = {"R2": np.percentile(r2s, [2.5, 97.5]).tolist(), "MAE": np.percentile(maes, [2.5, 97.5]).tolist(), "MAE_reduction": np.percentile(gain, [2.5, 97.5]).tolist()}
res = pd.DataFrame({t: {"model": final_name[t], "R2 before": before[t]["R2"], "R2 after": after[t]["R2"], "MAE before": before[t]["MAE"], "MAE after": after[t]["MAE"],
                        "MAE cut %": round((1 - after[t]["MAE"] / before[t]["MAE"]) * 100)} for t in TARGETS}).T
summary["after"], summary["bootstrap_ci"] = after, {"regression": boot}
display(res.round(3)); print(json.dumps(boot))

,model,R2 before,R2 after,MAE before,MAE after,MAE cut %
FAT_PCT,XGBoost,-0.003653,0.837272,7.198594,2.750306,62
LEAN_KG,Ridge,-0.000003,0.948655,10.484637,2.240133,79
ALM_KG,Ridge,-0.000018,0.927639,5.260064,1.339721,75
TRUNK_KG,Ridge,-0.000006,0.928083,5.103641,1.287848,75


{"FAT_PCT": {"R2": [0.8113570409332539, 0.8587216648117422], "MAE": [2.564568140973457, 2.9485307604814013], "MAE_reduction": [0.5837569333710388, 0.6494200190443725]}, "LEAN_KG": {"R2": [0.9374818586465526, 0.9577077986466067], "MAE": [2.076787086134078, 2.410582106876481], "MAE_reduction": [0.7656125611598739, 0.8059420362720058]}, "ALM_KG": {"R2": [0.9142822172644308, 0.9395337500666656], "MAE": [1.2394590989606185, 1.4430887482738084], "MAE_reduction": [0.7226530518164352, 0.7682672535491272]}, "TRUNK_KG": {"R2": [0.914266552397578, 0.9401644544048201], "MAE": [1.1931972459613636, 1.3871392677694314], "MAE_reduction": [0.7225531789055524, 0.7697629019747301]}}


## 7. ปรับให้ดีขึ้นทีละขั้น (Improvement ladder)

In [8]:
def fit_eval(t, feats, kind):
    m = ridge() if kind == "Ridge" else clone(models3[t]["XGBoost"])
    m.fit(train[feats], train[t]); return score(test[t], m.predict(test[feats]))
ladder = []
for t in TARGETS:
    ladder.append({"Target": t, "Step": "0 Before training (mean)", **before[t]})
    if t == "FAT_PCT": ladder.append({"Target": t, "Step": "0b Deurenberg formula", **deur_score})
    ladder.append({"Target": t, "Step": "1 Ridge: BMI + sex", **fit_eval(t, ["BMXBMI", "RIAGENDR"], "Ridge")})
    ladder.append({"Target": t, "Step": "2 Ridge: basic 5", **fit_eval(t, BASIC, "Ridge")})
    ladder.append({"Target": t, "Step": "3 Ridge: + circumferences", **fit_eval(t, FEAT, "Ridge")})
    ladder.append({"Target": t, "Step": "4 XGBoost tuned", **fit_eval(t, FEAT, "XGBoost")})
    ladder.append({"Target": t, "Step": "5 + lifestyle (chosen model)", **fit_eval(t, FEAT + LIFE, "Ridge" if final_name[t] == "Ridge" else "XGBoost")})
df_ladder = pd.DataFrame(ladder); summary["ladder"] = df_ladder.to_dict("records"); df_ladder.round(3)

,Target,Step,R2,MAE,RMSE
0,FAT_PCT,0 Before training (mean),-0.004,7.199,8.687
1,FAT_PCT,0b Deurenberg formula,0.529,4.612,5.949
2,FAT_PCT,1 Ridge: BMI + sex,0.752,3.400,4.318
3,FAT_PCT,2 Ridge: basic 5,0.762,3.325,4.233
4,FAT_PCT,3 Ridge: + circumferences,0.824,2.879,3.634
5,FAT_PCT,4 XGBoost tuned,0.837,2.750,3.498
6,FAT_PCT,5 + lifestyle (chosen model),0.838,2.747,3.494
7,LEAN_KG,0 Before training (mean),-0.000,10.485,12.789
8,LEAN_KG,1 Ridge: BMI + sex,0.801,4.500,5.712
9,LEAN_KG,2 Ridge: basic 5,0.936,2.500,3.227


## 8. ดูว่าตัวแปรช่วยแค่ไหน (Ablation และ Permutation importance)

In [9]:
def fit_final_on(t, feats):
    m = ridge() if final_name[t] == "Ridge" else clone(models3[t]["XGBoost"])
    return m.fit(train[feats], train[t])
sets = {"BMI + sex": ["BMXBMI", "RIAGENDR"], "basic 5": BASIC, "+ circumferences (10)": FEAT, "+ lifestyle (19)": FEAT + LIFE,
        "circumferences only": CIRC, "lifestyle only": LIFE}
abl = []
for t in TARGETS:
    row = {"Target": t}
    for name, feats in sets.items():
        row[name] = r2_score(test[t], fit_final_on(t, feats).predict(test[feats]))
    abl.append(row)
df_abl = pd.DataFrame(abl); display(df_abl.round(3))
perm = {}
for t in TARGETS:
    r = permutation_importance(final[t], test[FEAT], test[t], n_repeats=10, random_state=42, scoring="r2")
    perm[t] = pd.Series(r.importances_mean, index=FEAT).sort_values(ascending=False)
summary["ablation"], summary["permutation"] = df_abl.to_dict("records"), {t: perm[t].to_dict() for t in TARGETS}
pd.DataFrame(perm).round(3)

,Target,BMI + sex,basic 5,+ circumferences (10),+ lifestyle (19),circumferences only,lifestyle only
0,FAT_PCT,0.773,0.779,0.837,0.838,0.741,0.100
1,LEAN_KG,0.801,0.936,0.949,0.951,0.809,0.127
2,ALM_KG,0.754,0.894,0.928,0.930,0.813,0.136
3,TRUNK_KG,0.792,0.916,0.928,0.930,0.776,0.110


,FAT_PCT,LEAN_KG,ALM_KG,TRUNK_KG
BMXARMC,0.002,0.000,0.007,0.002
BMXARML,0.003,0.001,0.006,0.000
BMXBMI,0.024,0.007,0.010,0.003
BMXHIP,0.185,0.156,0.139,0.147
BMXHT,0.026,0.016,0.000,0.053
BMXLEG,0.003,-0.000,0.023,0.023
BMXWAIST,0.454,0.131,0.362,0.009
BMXWT,0.007,2.592,2.513,2.389
RIAGENDR,0.663,0.132,0.165,0.087
RIDAGEYR,0.003,0.000,0.000,0.000


## 9. Learning curve (Fat % XGBoost) และผลของขนาดข้อมูล

In [10]:
Xf, Xv = train_test_split(train, test_size=0.15, random_state=1)
m = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42, eval_metric="rmse")
m.fit(Xf[FEAT], Xf["FAT_PCT"], eval_set=[(Xf[FEAT], Xf["FAT_PCT"]), (Xv[FEAT], Xv["FAT_PCT"])], verbose=False)
r = m.evals_result(); trn, val = r["validation_0"]["rmse"], r["validation_1"]["rmse"]
base = float(np.sqrt(mean_squared_error(Xv["FAT_PCT"], np.full(len(Xv), Xf["FAT_PCT"].mean()))))
curves = {"FAT_PCT": {"train_rmse": trn, "val_rmse": val, "baseline_rmse": base}}
size_curve = {}
for t in TARGETS:
    sizes, trs, vas = learning_curve(ridge(), train[FEAT], train[t], train_sizes=[0.05, 0.1, 0.2, 0.4, 0.7, 1.0], cv=kf, scoring="r2")
    size_curve[t] = {"n": sizes.tolist(), "val_r2": vas.mean(axis=1).tolist()}
summary["curves"], summary["size_curve"] = curves, size_curve
print("baseline", round(base, 2), "| final train/val RMSE", round(trn[-1], 2), round(val[-1], 2))
{t: [round(x, 3) for x in v["val_r2"]] for t, v in size_curve.items()}

baseline 8.03 | final train/val RMSE 2.57 3.06


{'FAT_PCT': [0.808, 0.821, 0.821, 0.822, 0.824, 0.825],
 'LEAN_KG': [0.946, 0.951, 0.951, 0.952, 0.952, 0.953],
 'ALM_KG': [0.923, 0.93, 0.929, 0.93, 0.931, 0.931],
 'TRUNK_KG': [0.921, 0.929, 0.929, 0.931, 0.932, 0.933]}

## 10. MAE ก่อนและหลังเทรน แยกตามกลุ่ม

In [11]:
tt = test.copy()
for t in TARGETS: tt["pred_" + t] = preds[t]
tt["Sex"] = np.where(tt.RIAGENDR == 1, "Male", "Female")
tt["BMI group"] = pd.cut(tt.BMXBMI, [0, 25, 30, 100], labels=["<25", "25-30", ">=30"])
tt["Age group"] = pd.cut(tt.RIDAGEYR, [17, 30, 45, 60], labels=["18-30", "31-45", "46-59"])
rows = []
for col in ["Sex", "BMI group", "Age group"]:
    for g, d in tt.groupby(col, observed=True):
        r = {"Slice": col, "Group": g, "n": len(d)}
        for t in TARGETS:
            b = mean_absolute_error(d[t], np.full(len(d), train[t].mean())); a = mean_absolute_error(d[t], d["pred_" + t])
            r[t + " improved %"] = round((1 - a / b) * 100)
        rows.append(r)
df_slice = pd.DataFrame(rows); summary["slice"] = df_slice.to_dict("records"); df_slice

,Slice,Group,n,FAT_PCT improved %,LEAN_KG improved %,ALM_KG improved %,TRUNK_KG improved %
0,Sex,Female,250,65,80,75,77
1,Sex,Male,212,58,78,74,72
2,BMI group,<25,148,58,84,82,79
3,BMI group,25-30,136,56,74,71,67
4,BMI group,>=30,178,69,77,71,75
5,Age group,18-30,141,63,80,75,78
6,Age group,31-45,160,60,78,74,75
7,Age group,46-59,161,62,78,75,72


## 11. คัดกรองมวลกล้ามเนื้อต่ำ เทียบ 3 โมเดล

มี 2 นิยามที่ใช้กัน คือ **ALM/ส่วนสูง²** (ชาย<7.0, หญิง<5.5) กับ **ALM/BMI** (FNIH ชาย<0.789, หญิง<0.512) สองนิยามทับกันน้อยมาก เลยรายงานทั้งคู่

ศัพท์ที่ใช้เทียบ **Recall = Sensitivity** (จับคนที่มวลกล้ามเนื้อต่ำจริงได้กี่ %), **Precision = PPV** (ในคนที่ถูกเตือน ต่ำจริงกี่ %), **F1** = 2 × Precision × Recall / (Precision + Recall) เป็นค่าเฉลี่ยของสองตัวนี้ แต่ถ่วงเท่ากัน งานคัดกรองเน้นไม่ตกหล่น เลยดู Recall เป็นหลัก

In [12]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
def clf_lr(): return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=3000))
cl_rows, test_auc = [], {}
for lab in ["LOW_HT2", "LOW_FNIH"]:
    cands = {"Logistic Regression": clf_lr(), "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1),
             "XGBoost": XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")}
    for name, m in cands.items():
        oofp = cross_val_predict(m, train[FEAT], train[lab], cv=skf, method="predict_proba")[:, 1]
        pt = m.fit(train[FEAT], train[lab]).predict_proba(test[FEAT])[:, 1]
        cl_rows.append({"Label": lab, "Model": name, "CV AUC": roc_auc_score(train[lab], oofp), "CV PR-AUC": average_precision_score(train[lab], oofp),
                        "Test AUC": roc_auc_score(test[lab], pt), "Test PR-AUC": average_precision_score(test[lab], pt)})
    base = clf_lr().fit(train[["BMXBMI", "RIAGENDR"]], train[lab]).predict_proba(test[["BMXBMI", "RIAGENDR"]])[:, 1]
    test_auc[lab] = {"AUC_BMI_only": roc_auc_score(test[lab], base), "PR-AUC_BMI_only": average_precision_score(test[lab], base), "prevalence": float(test[lab].mean())}
df_cl = pd.DataFrame(cl_rows); summary["classification"], summary["classification_bmi_only"] = df_cl.to_dict("records"), test_auc
df_cl.round(3)

,Label,Model,CV AUC,CV PR-AUC,Test AUC,Test PR-AUC
0,LOW_HT2,Logistic Regression,0.955,0.744,0.966,0.799
1,LOW_HT2,Random Forest,0.941,0.693,0.939,0.695
2,LOW_HT2,XGBoost,0.939,0.694,0.955,0.755
3,LOW_FNIH,Logistic Regression,0.953,0.712,0.954,0.738
4,LOW_FNIH,Random Forest,0.933,0.632,0.906,0.653
5,LOW_FNIH,XGBoost,0.944,0.670,0.925,0.701


In [13]:
rng = np.random.default_rng(42); cls_ci, op_rows = {}, []
for lab in ["LOW_HT2", "LOW_FNIH"]:
    m = clf_lr().fit(train[FEAT], train[lab]); p = m.predict_proba(test[FEAT])[:, 1]
    oof = cross_val_predict(clf_lr(), train[FEAT], train[lab], cv=skf, method="predict_proba")[:, 1]
    pos = np.sort(oof[train[lab] == 1]); y = test[lab].values
    aucs, prs = [], []
    for _ in range(1000):
        i = rng.integers(0, len(test), len(test))
        if y[i].sum() == 0: continue
        aucs.append(roc_auc_score(y[i], p[i])); prs.append(average_precision_score(y[i], p[i]))
    cls_ci[lab] = {"AUC": np.percentile(aucs, [2.5, 97.5]).tolist(), "PR-AUC": np.percentile(prs, [2.5, 97.5]).tolist(), "n_pos_test": int(y.sum())}
    for sens in (0.9, 0.8):
        thr = float(pos[int(np.floor((1 - sens) * len(pos)))]); flag = p >= thr
        op_rows.append({"Label": lab, "Target sensitivity": sens, "Sensitivity": float((flag & (y == 1)).sum() / (y == 1).sum()),
                        "Referred %": float(flag.mean() * 100), "Specificity": float(((~flag) & (y == 0)).sum() / (y == 0).sum()),
                        "PPV": float((flag & (y == 1)).sum() / max(flag.sum(), 1))})
df_op = pd.DataFrame(op_rows); df_op["Recall"] = df_op["Sensitivity"]; df_op["Precision"] = df_op["PPV"]
df_op["F1"] = 2 * df_op["Precision"] * df_op["Recall"] / (df_op["Precision"] + df_op["Recall"]); summary["bootstrap_ci"]["classification"], summary["operating_points"] = cls_ci, df_op.to_dict("records")
display(df_op.round(3)); print(json.dumps(cls_ci))

,Label,Target sensitivity,Sensitivity,Referred %,Specificity,PPV,Recall,Precision,F1
0,LOW_HT2,0.9,0.915,18.182,0.901,0.512,0.915,0.512,0.656
1,LOW_HT2,0.8,0.851,14.502,0.935,0.597,0.851,0.597,0.702
2,LOW_FNIH,0.9,0.940,23.377,0.852,0.435,0.940,0.435,0.595
3,LOW_FNIH,0.8,0.800,16.450,0.913,0.526,0.800,0.526,0.635


{"LOW_HT2": {"AUC": [0.9446121807402451, 0.9806294431279621], "PR-AUC": [0.6893350287862688, 0.8805971205061917], "n_pos_test": 47}, "LOW_FNIH": {"AUC": [0.9265630618497741, 0.9769807592796875], "PR-AUC": [0.6044853165167797, 0.8538062163729925], "n_pos_test": 50}}


## 12. เทียบวิธีคัดกรอง 2 แบบ

Logistic Regression ได้ AUC สูงสุดทั้งสองนิยาม เลยเลือกเป็นตัวคัดกรอง แล้วลองเทียบว่าคัดกรองได้สองแบบ

- **แบบ A** ใช้ Ridge ประมาณ ALM จากค่าวัด 10 ตัว แล้วคำนวณ ALM/ส่วนสูง² เทียบเกณฑ์ตามเพศ
- **แบบ B (ตัวที่ใช้จริง)** ใช้ Logistic Regression ทายป้าย ต่ำ/ไม่ต่ำ ตรงๆ จากค่าวัดชุดเดียวกัน (ป้ายสร้างจาก ALM จริงของ DXA ตอนสอน)

เทียบ AUC บน Test พร้อม bootstrap 95% CI ของผลต่าง (B − A) และเทียบที่จำนวนคนที่แนะนำตรวจซ้ำเท่ากัน

In [14]:
alm_hat = final["ALM_KG"].predict(test[FEAT])
ratio_hat = {"LOW_HT2": alm_hat / (test.BMXHT.values / 100) ** 2, "LOW_FNIH": alm_hat / test.BMXBMI.values}
cut = {"LOW_HT2": test.RIAGENDR.map(rl.HT2_CUT).values, "LOW_FNIH": test.RIAGENDR.map(rl.FNIH_CUT).values}
rng = np.random.default_rng(0); ab_rows = []
for lab in ["LOW_HT2", "LOW_FNIH"]:
    y = test[lab].values
    sA = -(ratio_hat[lab] / cut[lab])
    sB = clf_lr().fit(train[FEAT], train[lab]).predict_proba(test[FEAT])[:, 1]
    diffs = []
    for _ in range(1000):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            diffs.append(roc_auc_score(y[i], sB[i]) - roc_auc_score(y[i], sA[i]))
    row = {"Label": lab, "n_pos": int(y.sum()), "AUC A": roc_auc_score(y, sA), "AUC B": roc_auc_score(y, sB),
           "B-A CI low": np.percentile(diffs, 2.5), "B-A CI high": np.percentile(diffs, 97.5),
           "A sens at plain cutoff": float((ratio_hat[lab] < cut[lab])[y == 1].mean())}
    for r in (0.14, 0.18, 0.23):
        for nm, s in (("A", sA), ("B", sB)):
            flag = s >= np.quantile(s, 1 - r)
            row[f"{nm} sens @{int(r * 100)}%"] = float(flag[y == 1].mean())
    ab_rows.append(row)
df_ab = pd.DataFrame(ab_rows); summary["screening_A_vs_B"] = df_ab.to_dict("records")
df_ab.round(3).T

,0,1
Label,LOW_HT2,LOW_FNIH
n_pos,47,50
AUC A,0.964,0.951
AUC B,0.966,0.954
B-A CI low,-0.005,-0.013
B-A CI high,0.009,0.024
A sens at plain cutoff,0.532,0.6
A sens @14%,0.809,0.78
B sens @14%,0.851,0.8
A sens @18%,0.894,0.82
